# rank0-only-side-effects — worked example 2: Log epoch metrics to wandb on rank 0 only

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank0-only-side-effects`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Logging tools like wandb create a new run object for each process that calls `wandb.init()`. In distributed training, calling `wandb.log(metrics)` on every rank would create `world_size` duplicate run entries with the same metrics. The correct pattern is to initialize and log only on rank 0, while all other ranks skip the wandb calls entirely. The metrics themselves may be gathered from all ranks first, then logged once by rank 0.

## Worked solution

**Step 1 — Gather metrics from all ranks (simulation).** In real code, you would use `dist.all_reduce` to sum or average per-rank metrics. Here we simulate by accepting a pre-gathered loss value.

**Step 2 — Guard wandb.init on rank 0.** `if rank == 0: run = wandb_module.init(project='my-proj')`. Non-zero ranks skip this and `run` stays None.

**Step 3 — Guard wandb.log on rank 0.** `if rank == 0: run.log({'train/loss': mean_loss})`. The metrics are logged once regardless of `world_size`.

**Step 4 — Verify.** We use a mock wandb module and check that `init` and `log` were each called exactly once, not `world_size` times.

In [ ]:
from dataclasses import dataclass, field
from typing import Any

@dataclass
class MockRun:
    logs: list = field(default_factory=list)
    def log(self, metrics):
        self.logs.append(metrics)

class MockWandb:
    def __init__(self):
        self.init_calls = 0
        self.run = None
    def init(self, **kwargs):
        self.init_calls += 1
        self.run = MockRun()
        return self.run

def epoch_end_log(rank: int, world_size: int, mean_loss: float, wandb_module) -> None:
    """Log epoch metrics; only rank 0 touches wandb."""
    if rank == 0:
        run = wandb_module.init(project='dist-training')
        run.log({'train/loss': mean_loss, 'epoch': 1})

# Simulate 3 ranks
fake_wandb = MockWandb()
for rank in range(3):
    epoch_end_log(rank=rank, world_size=3, mean_loss=0.42, wandb_module=fake_wandb)

print(f'wandb.init calls: {fake_wandb.init_calls} (expect 1)')
print(f'log calls:        {len(fake_wandb.run.logs)} (expect 1)')
print(f'Logged metric:    {fake_wandb.run.logs[0]}')